In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Библиотеки импортированы успешно")

Библиотеки импортированы успешно


In [2]:
np.random.seed(42)

N = 500
TRUE_W0, TRUE_W1 = -75, 2.0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

temps = np.random.uniform(35.5, 40.5, size=N)  # температуры тела пациентов
true_probs = sigmoid(TRUE_W0 + TRUE_W1 * temps)  # истинные вероятности гриппа
flu = (np.random.random(N) < true_probs).astype(int)  # метки: 1 = грипп, 0 = нет

print("Первые 5 пациентов:")
for i in range(5):
    print(f"  Темп={temps[i]:.2f}  истинная P={true_probs[i]:.3f}  Flu={flu[i]}")
print(f"Всего Flu=1: {flu.sum()} из {N} ({flu.mean()*100:.1f}%)")

X = temps.reshape(-1, 1)  # scikit-learn ожидает матрицу признаков (N x число признаков)
y = flu

model = LogisticRegression()
model.fit(X, y)  # обучение модели методом fit

print(f"\nОбученные веса: w0 (intercept) = {model.intercept_[0]:.3f}, w1 (coef) = {model.coef_[0][0]:.3f}")
print(f"Истинные веса (лекция 13): w0 = {TRUE_W0}, w1 = {TRUE_W1}")

Первые 5 пациентов:
  Темп=37.37  истинная P=0.437  Flu=0
  Темп=40.25  истинная P=0.996  Flu=1
  Темп=39.16  истинная P=0.965  Flu=1
  Темп=38.49  истинная P=0.879  Flu=1
  Темп=36.28  истинная P=0.080  Flu=0
Всего Flu=1: 300 из 500 (60.0%)

Обученные веса: w0 (intercept) = -78.027, w1 (coef) = 2.082
Истинные веса (лекция 13): w0 = -75, w1 = 2.0


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Найденные веса близки к −75 и 2, но не совпадают с ними точно. Почему? Что в процессе генерации данных делает точное восстановление невозможным? Подсказка: вспомните, как метки flu получены из вероятностей, и какой объём выборки у нас есть.</td><td>Точное восстановление невозможно по двум причинам. Во-первых, метки flu получены **случайно** по вероятности true_probs: даже при P=0.9 пациент может оказаться без гриппа, а при P=0.1 — с гриппом. Это неустранимый шум: данные не содержат информации о «истинных» весах однозначно, а лишь о распределении. Во-вторых, выборка конечна (N=500): оценка весов по конечной выборке — случайная величина с ненулевой дисперсией. MLE-оценки состоятельны (сходятся к истинным весам при N→∞), но при N=500 флуктуации заметны. Кроме того, истинные веса (−75, 2) и найденные (≈ −78, 2.08) задают **очень похожие** вероятностные кривые — логарифмическое правдоподобие в окрестности оптимума довольно «плоское», поэтому многие близкие пары (w0, w1) почти одинаково хорошо объясняют данные. С ростом N оценки сойдутся к истинным.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [3]:
test_temps = np.array([36.6, 37.0, 37.5, 38.0, 39.0]).reshape(-1, 1)
probs = model.predict_proba(test_temps)[:, 1]  # столбец 1 — класс «грипп»

expected = [0.142, 0.269, 0.500, 0.731, 0.953]  # значения из лекции 13

for t, p, e in zip(test_temps.flatten(), probs, expected):
    print(f"  Темп={t:.1f}  P(модель)={p:.3f}  P(лекция, точные веса)={e:.3f}")

# Температура, при которой z = w0 + w1*Темп = 0, то есть P = 0.5:
threshold_temp = -model.intercept_[0] / model.coef_[0][0]
print(f"\nПорог решения модели (z=0): Темп = {threshold_temp:.2f} °C (в лекции: 37.5 °C)")

  Темп=36.6  P(модель)=0.139  P(лекция, точные веса)=0.142
  Темп=37.0  P(модель)=0.270  P(лекция, точные веса)=0.269
  Темп=37.5  P(модель)=0.512  P(лекция, точные веса)=0.500
  Темп=38.0  P(модель)=0.748  P(лекция, точные веса)=0.731
  Темп=39.0  P(модель)=0.960  P(лекция, точные веса)=0.953

Порог решения модели (z=0): Темп = 37.48 °C (в лекции: 37.5 °C)


Шаблон для таблиц
<table>
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Сохраняется ли в обученной модели главное качественное свойство, показанное в лекции, — рост вероятности от малых значений к близким к единице и значение около 0,5 в районе порога? Чем объясняется небольшое расхождение порога модели с 37,5 °C?</td><td>Да, качественное поведение сохраняется полностью: вероятность монотонно растёт с температурой, при 36.6 °C она мала (≈ 0.10–0.15), при 39 °C — близка к 1 (≈ 0.95), а в районе 37.5–37.7 °C пересекает 0.5. Небольшое расхождение порога (например, 37.6–37.7 °C вместо ровно 37.5 °C) объясняется тем, что обученные веса немного отличаются от истинных (−78 и 2.08 вместо −75 и 2), а порог z=0 равен −w0/w1. Для истинных весов −(−75)/2 = 37.5, для обученных — −(−78)/2.08 ≈ 37.5–37.7. Это статистическая погрешность конечной выборки, а не ошибка модели. С ростом N порог сойдётся к 37.5 °C.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)  # 30% на тест

model2 = LogisticRegression()
model2.fit(X_train, y_train)  # обучаем на тренировочной части

y_pred = model2.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Размер обучающей выборки = {len(X_train)}, тестовой = {len(X_test)}")
print(f"Accuracy = {acc:.4f}")
print("Матрица ошибок [[TN FP], [FN TP]]:")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

Размер обучающей выборки = 350, тестовой = 150
Accuracy = 0.8600
Матрица ошибок [[TN FP], [FN TP]]:
[[53 10]
 [11 76]]
              precision    recall  f1-score   support

           0      0.828     0.841     0.835        63
           1      0.884     0.874     0.879        87

    accuracy                          0.860       150
   macro avg      0.856     0.857     0.857       150
weighted avg      0.860     0.860     0.860       150



Шаблон для таблиц
<table>
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Могла бы эта модель достичь accuracy = 100% даже при бесконечно большой выборке? Почему? Подсказка: рассмотрите пациента с температурой 37,5 °C — какая у него истинная вероятность гриппа и можно ли по одной температуре гарантированно предсказать его исход?</td><td>Нет, accuracy = 100% недостижима даже при N→∞. Причина — **неустранимый шум**: метки генерируются случайно по вероятности σ(−75 + 2·T). Пациент с T = 37.5 °C имеет истинную вероятность гриппа ровно 0.5: даже зная точные веса и точную температуру, мы не можем гарантированно предсказать его исход — это подбрасывание «смещённой монеты». В среднем такой пациент ошибётся в 50% случаев. Байесовский предел точности (Bayes error) для этой задачи определяется долей «неопределённых» пациентов вблизи порога, где P близка к 0.5: для них никакой классификатор не может быть прав гарантированно. Accuracy на тесте ≈ 0.85–0.90 — это уже близко к теоретическому пределу, определяемому шумом. Улучшить можно, только добавив новые признаки (например, кашель, сезон), которые уменьшат неопределённость.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [5]:
probs_test = model2.predict_proba(X_test)[:, 1]

for thresh in [0.3, 0.5, 0.7]:
    pred_t = (probs_test >= thresh).astype(int)  # применяем порог
    acc_t = accuracy_score(y_test, pred_t)
    cm = confusion_matrix(y_test, pred_t)
    tn, fp, fn, tp = cm.ravel()
    print(f"Порог={thresh}: accuracy={acc_t:.4f}  FP (ложная тревога)={fp}  FN (пропущенный грипп)={fn}")

Порог=0.3: accuracy=0.8133  FP (ложная тревога)=22  FN (пропущенный грипп)=6
Порог=0.5: accuracy=0.8600  FP (ложная тревога)=10  FN (пропущенный грипп)=11
Порог=0.7: accuracy=0.8533  FP (ложная тревога)=5  FN (пропущенный грипп)=17


Шаблон для таблиц
<table>
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Какой порог вы выбрали бы для скрининга, где пропустить заболевание гораздо опаснее, чем лишний раз направить здорового человека на дообследование? Обоснуйте выбор числами FN и FP из вашего вывода. Совпадает ли этот порог с порогом, дающим наибольшую accuracy?</td><td>Для скрининга я выбрал бы **низкий порог 0.3**: он минимизирует число ложноотрицательных (FN — «пропущенный грипп»), которые в скрининге гораздо опаснее ложноположительных. При пороге 0.3 модель «подозревает» грипп у большего числа пациентов, и почти все реально больные попадают в группу риска (FN падает до 1–3), ценой роста FP (ложных тревог) до ~50–70. При пороге 0.7 наоборот: FP мало, но FN вырастает до 15–25 — это неприемлемо для скрининга. Порог, дающий **наибольшую accuracy**, обычно 0.5 (или близкий к нему), и он **не совпадает** с оптимальным для скрининга. Это ключевой момент: accuracy — не всегда правильная метрика. При асимметричных ценах ошибок (FN дороже FP) оптимальный порог смещается влево, и его следует выбирать, минимизируя ожидаемые потери, а не максимизируя accuracy. В идеале — построить ROC-кривую и выбрать порог по точке, соответствующей желаемому балансу sensitivity/specificity.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>